#### Generador de tickets por negocio por fecha

In [417]:
#motor semántico + gobernanza + aprendizaje incremental
import os
import pandas as pd
from sqlalchemy import create_engine
from urllib.parse import quote_plus
from dotenv import load_dotenv


# Cargar variables de entorno
load_dotenv()

# === DATOS DE CONEXIÓN ===
DB_USER = os.getenv("DB_USER")
DB_PASS = quote_plus(os.getenv("DB_PASS", "")) # se codifica aquí
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

# === ENGINE ===
engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}",
    pool_pre_ping=True
)

# === PRUEBA DE CONEXIÓN ===
with engine.connect() as conn:
    print("✅ Conexión exitosa a MySQL")



✅ Conexión exitosa a MySQL


In [418]:
# === CARGAR TABLA ===
categories = pd.read_sql("""
SELECT id, name
FROM categories
""", engine)

# === LIMPIAR DIRECTO EN name ===
categories["name"] = (
    categories["name"]
    .astype(str)
    .str.strip()
    .str.replace(r'^\s*\d{1,4}\s*[-._\)\]]\s*', '', regex=True)  # quita 01-, 001-, 1-, 02)
    .str.replace(r'^\s*\d{1,4}\s+', '', regex=True)              # quita "01 PIZZA"
    .str.strip()
)

categories.head()

,id,name
0,31,TACOS LOS LICENCIADOS
1,33,PIZZA CENTRAL
2,34,DESAYUNOS GAHORY
3,35,EXPRESSO CENTRAL
4,36,"HAMBURGUESAS ""EL GÜERO"""


In [419]:
products = pd.read_sql("""
SELECT
  id,
  name,
  price,
  image1,
  id_category
FROM products
""", engine)

products.head()

,id,name,price,image1,id_category
0,38,ENCHILADAS(DESAYUNOS GAHORY),70.0,https://firebasestorage.googleapis.com/v0/b/te...,34
1,39,ENCHILADAS NORTEÑA(DESAYUNOS GAHORY),85.0,https://firebasestorage.googleapis.com/v0/b/te...,34
2,40,ENCHILADAS SUIZO(DESAYUNOS GAHORY),95.0,https://firebasestorage.googleapis.com/v0/b/te...,34
3,41,ENCHILADAS DELICIA(DESAYUNOS GAHORY),85.0,https://firebasestorage.googleapis.com/v0/b/te...,34
4,42,ENCHILADAS PAISA(DESAYUNOS GAHORY),85.0,https://firebasestorage.googleapis.com/v0/b/te...,34


In [420]:
orders = pd.read_sql("""
SELECT
  id,
  id_client,
  created_at,
  status,
  updated_at
FROM orders
WHERE status = 'COMPLETADO'
""", engine)

orders.head()

,id,id_client,created_at,status,updated_at
0,87,75,2026-01-19 12:25:30,COMPLETADO,2025-10-20 13:35:27
1,88,102,2026-01-19 12:25:34,COMPLETADO,2025-10-21 13:36:08
2,89,107,2026-01-19 12:25:38,COMPLETADO,2025-10-21 18:42:50
3,90,85,2026-01-19 12:25:42,COMPLETADO,2025-10-21 20:46:38
4,92,113,2026-01-19 12:25:51,COMPLETADO,2025-10-23 19:19:06


In [421]:
order_has_products = pd.read_sql("""
SELECT
  ohp.id,
  ohp.id_order,
  ohp.id_product,
  ohp.quantity,
  ohp.created_at
FROM order_has_products ohp
INNER JOIN orders o ON o.id = ohp.id_order
WHERE o.status = 'COMPLETADO'
""", engine)

order_has_products.head()

,id,id_order,id_product,quantity,created_at
0,13,87,154,1,2025-10-20 12:40:11
1,14,87,154,1,2025-10-20 12:40:11
2,15,88,39,1,2025-10-21 12:57:03
3,16,88,38,1,2025-10-21 12:57:03
4,17,89,269,1,2025-10-21 17:37:02


In [422]:
import pandas as pd
from datetime import date

START_DATE = "2026-01-30"           # ← aquí pones el día que quieres
END_DATE = date.today().isoformat() # ← hoy automático

params = {"start": START_DATE, "end": END_DATE}  # ✅ IMPORTANTÍSIMO

print("Rango:", START_DATE, "→", END_DATE)




Rango: 2026-01-30 → 2026-02-05


In [423]:
query = """
SELECT
  c.id AS id_category,
  c.name AS category_name,
  GROUP_CONCAT(DISTINCT o.id ORDER BY o.id) AS order_ids,
  SUM(ohp.quantity) AS units_sold,
  ROUND(SUM(ohp.quantity * p.price), 2) AS gross_sales,
  COUNT(DISTINCT o.id) AS orders_count
FROM orders o
JOIN order_has_products ohp ON o.id = ohp.id_order
JOIN products p ON p.id = ohp.id_product
JOIN categories c ON c.id = p.id_category
WHERE o.created_at >= %(start)s
  AND o.created_at < DATE_ADD(%(end)s, INTERVAL 1 DAY)
  AND TRIM(UPPER(o.status)) = 'ENTREGADO'
GROUP BY c.id, c.name
ORDER BY gross_sales DESC;
"""

df = pd.read_sql(query, engine, params=params).reset_index(drop=True)



In [424]:
# LIMPIAR category_name
df["category_name"] = (
    df["category_name"]
    .astype(str)
    .str.strip()
    .str.replace(r'^\s*\d{1,4}\s*[-._\)\]]\s*', '', regex=True)
    .str.replace(r'^\s*\d{1,4}\s+', '', regex=True)
    .str.strip()
)

# 🔢 units_sold sin decimales
df["units_sold"] = df["units_sold"].astype(int)

# 💰 gross_sales formato dinero
df["gross_sales"] = df["gross_sales"].astype(float)
df["gross_sales"] = df["gross_sales"].apply(lambda x: f"${x:,.2f}")

print(df)

    id_category            category_name                        order_ids  \
0            71   K-RAMEEN KOREAN SNACKS  511,514,521,522,523,524,525,526   
1            42              LA BARRANCA              533,534,545,546,550   
2            59              MERAKI CAFÉ                      508,527,539   
3            56          EII MAR CONEJOS                      517,519,536   
4            69               FRITALITAS          512,515,529,531,542,544   
5            38    NONIS FUENTE DE SODAS                  518,548,549,551   
6            33            PIZZA CENTRAL                              552   
7            39              TACOS JESSY                          530,540   
8            72    AIRBONE GRILL&BURGERS                      513,515,544   
9            60           ANTOJO BURGERS                              528   
10           48           LA CHILAQUERIA                              506   
11           34         DESAYUNOS GAHORY                              532   

In [425]:
############3

In [426]:
import pandas as pd

# params debe existir así:
# params = {"start": START_DATE, "end": END_DATE}

detail_query = """
SELECT
  c.id AS id_category,
  c.name AS category_name,
  o.id AS id_order,
  o.created_at,
  p.id AS id_product,
  p.name AS product_name,
  ohp.quantity,
  p.price
FROM orders o
JOIN order_has_products ohp ON ohp.id_order = o.id
JOIN products p ON p.id = ohp.id_product
JOIN categories c ON c.id = p.id_category
WHERE o.created_at >= %(start)s
  AND o.created_at < DATE_ADD(%(end)s, INTERVAL 1 DAY)
  AND TRIM(UPPER(o.status)) = 'ENTREGADO'
ORDER BY c.name, o.id, p.name;
"""

df_detail = pd.read_sql(detail_query, engine, params=params)
df_detail.head()

,id_category,category_name,id_order,created_at,id_product,product_name,quantity,price
0,73,01-ALQUIMIA DULCE,510,2026-01-30 17:56:30,738,FLAN NAPOLITANO (ALQUIMIA DULCE) NO DISPONIBLE,1,40.0
1,73,01-ALQUIMIA DULCE,520,2026-01-31 18:02:49,740,CARLOTA DE LIMÓN (ALQUIMIA DULCE),1,45.0
2,73,01-ALQUIMIA DULCE,520,2026-01-31 18:02:49,738,FLAN NAPOLITANO (ALQUIMIA DULCE) NO DISPONIBLE,1,40.0
3,59,01-MERAKI CAFÉ,508,2026-01-30 16:35:47,440,PASTA ALFREDO CON POLLO (MERAKI CAFE),1,155.0
4,59,01-MERAKI CAFÉ,527,2026-01-31 21:19:00,435,DEDOS DE QUESO (MERAKI CAFE),1,79.0


In [427]:
import numpy as np

# limpiar prefijos tipo 01- / 02- / 001- / 01 MERAKI
df_detail["category_name"] = (
    df_detail["category_name"].astype(str).str.strip()
    .str.replace(r'^\s*\d{1,4}\s*[-._\)\]]\s*', '', regex=True)
    .str.replace(r'^\s*\d{1,4}\s+', '', regex=True)
    .str.strip()
)

# asegurar tipos numéricos
df_detail["quantity"] = pd.to_numeric(df_detail["quantity"], errors="coerce").fillna(0).astype(int)
df_detail["price"] = pd.to_numeric(df_detail["price"], errors="coerce").fillna(0.0)

df_detail["line_total"] = df_detail["quantity"] * df_detail["price"]
df_detail.head()

,id_category,category_name,id_order,created_at,id_product,product_name,quantity,price,line_total
0,73,ALQUIMIA DULCE,510,2026-01-30 17:56:30,738,FLAN NAPOLITANO (ALQUIMIA DULCE) NO DISPONIBLE,1,40.0,40.0
1,73,ALQUIMIA DULCE,520,2026-01-31 18:02:49,740,CARLOTA DE LIMÓN (ALQUIMIA DULCE),1,45.0,45.0
2,73,ALQUIMIA DULCE,520,2026-01-31 18:02:49,738,FLAN NAPOLITANO (ALQUIMIA DULCE) NO DISPONIBLE,1,40.0,40.0
3,59,MERAKI CAFÉ,508,2026-01-30 16:35:47,440,PASTA ALFREDO CON POLLO (MERAKI CAFE),1,155.0,155.0
4,59,MERAKI CAFÉ,527,2026-01-31 21:19:00,435,DEDOS DE QUESO (MERAKI CAFE),1,79.0,79.0


In [428]:
import os, re, textwrap
from datetime import datetime
from reportlab.lib.pagesizes import letter
from reportlab.lib.units import cm
from reportlab.lib import colors
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, PageBreak
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle

OUT_DIR = "pdf_por_negocio_desglosado"
os.makedirs(OUT_DIR, exist_ok=True)

styles = getSampleStyleSheet()
title_style = ParagraphStyle("t", parent=styles["Title"], fontSize=18, leading=22)
h_style     = ParagraphStyle("h", parent=styles["Heading2"], fontSize=13, leading=16, spaceBefore=10)
p_style     = ParagraphStyle("p", parent=styles["BodyText"], fontSize=11, leading=14)

def safe_filename(s: str) -> str:
    s = str(s).strip()
    s = re.sub(r'[\\/*?:"<>|]', "", s)
    s = re.sub(r"\s+", " ", s)
    return s[:120]

def money(x: float) -> str:
    return f"${x:,.2f}"

def build_header(canvas, doc, negocio):
    canvas.saveState()
    canvas.setFont("Helvetica", 9)
    canvas.drawString(2*cm, 27.8*cm, f"DOBLEU | Corte por negocio")
    canvas.drawRightString(20.5*cm, 27.8*cm, f"Negocio: {negocio}")
    canvas.drawString(2*cm, 27.2*cm, f"Periodo: {params['start']} a {params['end']}")
    canvas.drawRightString(20.5*cm, 27.2*cm, datetime.now().strftime("%Y-%m-%d %H:%M"))
    canvas.restoreState()

pdf_paths = []

# agrupar por negocio
for negocio, g_neg in df_detail.groupby("category_name", sort=True):
    filename = safe_filename(negocio)
    pdf_path = os.path.join(OUT_DIR, f"{filename}.pdf")

    doc = SimpleDocTemplate(
        pdf_path,
        pagesize=letter,
        leftMargin=2*cm, rightMargin=2*cm,
        topMargin=3*cm, bottomMargin=2*cm
    )

    story = []
    story.append(Paragraph(negocio, title_style))
    story.append(Spacer(1, 8))

    # total negocio (en el periodo)
    total_negocio = float(g_neg["line_total"].sum())
    total_ordenes = g_neg["id_order"].nunique()
    story.append(Paragraph(f"<b>Total de ventas:</b> {money(total_negocio)}", p_style))
    story.append(Paragraph(f"<b>Total de órdenes (tickets):</b> {total_ordenes}", p_style))
    story.append(Spacer(1, 10))

    # por ticket
    for id_order, g_ord in g_neg.groupby("id_order", sort=True):
        story.append(Paragraph(f"Ticket #{int(id_order)}", h_style))

        # tabla de productos
        data = [["Producto", "Qty", "Precio", "Subtotal"]]
        for _, r in g_ord.iterrows():
            data.append([
                str(r["product_name"]),
                str(int(r["quantity"])),
                money(float(r["price"])),
                money(float(r["line_total"]))
            ])

        t = Table(data, colWidths=[9.5*cm, 2*cm, 3.5*cm, 3.5*cm])
        t.setStyle(TableStyle([
            ("BACKGROUND", (0,0), (-1,0), colors.lightgrey),
            ("TEXTCOLOR", (0,0), (-1,0), colors.black),
            ("FONTNAME", (0,0), (-1,0), "Helvetica-Bold"),
            ("FONTNAME", (0,1), (-1,-1), "Helvetica"),
            ("FONTSIZE", (0,0), (-1,-1), 10),
            ("GRID", (0,0), (-1,-1), 0.25, colors.grey),
            ("ALIGN", (1,1), (1,-1), "CENTER"),
            ("ALIGN", (2,1), (3,-1), "RIGHT"),
            ("VALIGN", (0,0), (-1,-1), "MIDDLE"),
        ]))
        story.append(t)

        # total del ticket
        total_ticket = float(g_ord["line_total"].sum())
        story.append(Spacer(1, 6))
        story.append(Paragraph(f"<b>Total ticket #{int(id_order)}:</b> {money(total_ticket)}", p_style))
        story.append(Spacer(1, 10))

    doc.build(
        story,
        onFirstPage=lambda c, d, n=negocio: build_header(c, d, n),
        onLaterPages=lambda c, d, n=negocio: build_header(c, d, n),
    )

    pdf_paths.append(pdf_path)

print(f"✅ PDFs generados: {len(pdf_paths)} en: {OUT_DIR}")

✅ PDFs generados: 17 en: pdf_por_negocio_desglosado
